In [1]:
# ── Portable environment bootstrap · local ↔ Kaggle ──────────────────────────
# No-op on my machine (the notebook already runs from notebooks/, so ../data and
# ../reports resolve). On Kaggle it rebuilds that layout: it finds the cleaned
# dataset file whatever its name/compression (Kaggle may rename or extract it),
# detects the real format from the file's magic bytes, and materialises it at
# ../data/processed/df_clean.csv — so every relative path below keeps working.
import os, glob, gzip, zipfile, shutil
from pathlib import Path

if Path('/kaggle/input').exists():
    work   = Path('/kaggle/working')
    nb_dir = work / 'notebooks'
    nb_dir.mkdir(exist_ok=True)
    os.chdir(nb_dir)                                   # now '..' == /kaggle/working
    proc = work / 'data' / 'processed'
    proc.mkdir(parents=True, exist_ok=True)
    (work / 'reports' / 'figures').mkdir(parents=True, exist_ok=True)

    target = proc / 'df_clean.csv'
    if not target.exists():
        cands = [c for c in glob.glob('/kaggle/input/**/df_clean*', recursive=True)
                 if os.path.isfile(c)]
        if not cands:
            raise FileNotFoundError(
                "No df_clean.* file in the attached dataset — attach the Cyclistic dataset.")
        # prefer a real .csv, else the largest candidate
        csvs = [c for c in cands if c.lower().endswith('.csv')]
        src  = csvs[0] if csvs else max(cands, key=os.path.getsize)

        with open(src, 'rb') as f:
            magic = f.read(2)
        if magic == b'\x1f\x8b':                       # gzip
            with gzip.open(src, 'rb') as fi, open(target, 'wb') as fo:
                shutil.copyfileobj(fi, fo)
        elif magic == b'PK':                            # real zip archive
            with zipfile.ZipFile(src) as z:
                inner = [n for n in z.namelist() if n.lower().endswith('.csv')] or z.namelist()
                with z.open(inner[0]) as fi, open(target, 'wb') as fo:
                    shutil.copyfileobj(fi, fo)
        else:                                           # already plain-text CSV
            os.symlink(src, target)
        print(f"Kaggle detected — df_clean.csv ready (from {os.path.basename(src)})")
    else:
        print("Kaggle detected — df_clean.csv already present.")
else:
    print("Local environment — using the existing ../data and ../reports layout.")


Local environment — using the existing ../data and ../reports layout.


# Table of Contents <a id='toc'></a>

1. [Setup and Data Loading](#setup)
2. [Data Exploration](#explore)
   - 2.1 [Schema, Data Types and Dimensions](#schema)
   - 2.2 [Coordinate Bounds Validation](#coords)
   - 2.3 [Missing Values Overview](#missing)
3. [Data Cleaning](#cleaning)
   - 3.1 [Removing Rows with Missing GPS Coordinates](#gps)
   - 3.2 [Electric Bikes and Missing Station Names](#ebike)
   - 3.3 [Standardizing Categorical Values](#standardize)
   - 3.4 [Deduplication Check](#dedup)
   - 3.5 [Timestamp Consistency](#timestamps)
4. [Cleaning Summary](#summary)
5. [Conclusions](#conclusions)
6. [Saving Outputs](#saving)

---
# 1. Setup and Data Loading <a id='setup'></a>
[↑ back to top](#toc)

**Source:** [Divvy Trip Data](https://divvy-tripdata.s3.amazonaws.com/index.html) — 12 monthly CSV files (June 2025 – May 2026).

All files are loaded and concatenated into a single DataFrame.  
A column-schema consistency check is applied before merging: any file with a different structure is isolated for inspection rather than silently included.

In [2]:
import numpy as np
import pandas as pd
import glob
import os

In [3]:
# Load all monthly CSV files; verify consistent column schema before merging
files = sorted(glob.glob('../data/raw/trips/*.csv'))

KAGGLE_MODE = not files  # True on Kaggle where raw trips aren't available

if KAGGLE_MODE:
    # Raw trip files aren't available on Kaggle — load df_clean.csv to demonstrate
    # the pipeline logic on real data
    print("ℹ️  Kaggle mode: raw trip files not available.")
    print("   Loading df_clean.csv to demonstrate the pipeline on processed data.\n")
    df = pd.read_csv('../data/processed/df_clean.csv')
    n_raw = len(df)
    print(f"Rows loaded from df_clean.csv: {n_raw:,}")
else:
    dfs        = []   # files whose schema matches the reference
    dfs_other  = []   # files with a different schema (flagged for inspection)
    reference_columns = None

    for path in files:
        df_temp   = pd.read_csv(path)
        temp_cols = set(df_temp.columns)
        if reference_columns is None:
            reference_columns = temp_cols
            dfs.append(df_temp)
        elif temp_cols == reference_columns:
            dfs.append(df_temp)
        else:
            dfs_other.append(df_temp)

    df    = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    n_raw = len(df)

    print(f"Files loaded       : {len(dfs)}")
    print(f"Schema mismatches  : {len(dfs_other)}")
    print(f"Total rows         : {n_raw:,}")

Files loaded       : 12
Schema mismatches  : 0
Total rows         : 5,848,703


In [4]:
if not KAGGLE_MODE:
    # Report any files that did not match the reference column schema
    if dfs_other:
        print(f"Files with different schema ({len(dfs_other)}):")
        for d in dfs_other:
            print("  Columns found:", list(d.columns))
    else:
        print("All files share the same column schema. ✓")
else:
    print("Schema check skipped in Kaggle mode (loading pre-cleaned data).")

All files share the same column schema. ✓


---
# 2. Data Exploration <a id='explore'></a>
[↑ back to top](#toc)

Before cleaning, we inspect the dataset to understand its structure, value distributions, geographic coverage, and completeness.  
This baseline exploration informs every subsequent cleaning decision.

## 2.1 Schema, Data Types and Dimensions <a id='schema'></a>

Verify that categorical columns (`rideable_type`, `member_casual`) contain only expected values and check for any unexpected data types.

In [5]:
print("Bike types      :", df['rideable_type'].unique())
print("Customer types  :", df['member_casual'].unique())
print()
df.info()
print()
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Bike types      : ['electric_bike' 'classic_bike']
Customer types  : ['member' 'casual']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5848703 entries, 0 to 5848702
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             object 
 1   rideable_type       object 
 2   started_at          object 
 3   ended_at            object 
 4   start_station_name  object 
 5   start_station_id    object 
 6   end_station_name    object 
 7   end_station_id      object 
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       object 
dtypes: float64(4), object(9)
memory usage: 580.1+ MB

Shape: 5,848,703 rows × 13 columns


## 2.2 Coordinate Bounds Validation <a id='coords'></a>

Confirm that all trip coordinates fall within the expected geographic bounds for Chicago and surrounding areas  
(latitude ≈ 41.6°–42.1°N, longitude ≈ -87.9°–-87.5°W).  
Any records with coordinates far outside this range would indicate data errors or test rides.

In [6]:
coord_cols = ['start_lat', 'start_lng', 'end_lat', 'end_lng']
print(df[coord_cols].describe().round(4))

          start_lat     start_lng       end_lat       end_lng
count  5.848703e+06  5.848703e+06  5.842807e+06  5.842807e+06
mean   4.190420e+01 -8.764660e+01  4.190460e+01 -8.764700e+01
std    4.450000e-02  2.710000e-02  4.460000e-02  2.730000e-02
min    4.164850e+01 -8.789000e+01  4.149000e+01 -8.810000e+01
25%    4.188240e+01 -8.766000e+01  4.188280e+01 -8.766000e+01
50%    4.189960e+01 -8.764180e+01  4.190000e+01 -8.764290e+01
75%    4.193000e+01 -8.763000e+01  4.193000e+01 -8.763000e+01
max    4.207000e+01 -8.752000e+01  4.221000e+01 -8.742000e+01


## 2.3 Missing Values Overview <a id='missing'></a>

A column-level null count to identify which fields need attention.  
Some nulls (e.g. station names for electric bikes) are by design and **will not** be removed — they are explained in Section 3.2.

In [7]:
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
report      = pd.DataFrame({'missing_count': missing, 'missing_pct_%': missing_pct})
print(report[report['missing_count'] > 0].to_string())

                    missing_count  missing_pct_%
start_station_name        1249667          21.37
start_station_id          1249667          21.37
end_station_name          1314334          22.47
end_station_id            1314334          22.47
end_lat                      5896           0.10
end_lng                      5896           0.10


---
# 3. Data Cleaning <a id='cleaning'></a>
[↑ back to top](#toc)

The cleaning pipeline addresses five distinct issues, each handled in a separate sub-section.  
All steps are applied sequentially; row counts are tracked to quantify the impact of each decision.

## 3.1 Removing Rows with Missing GPS Coordinates <a id='gps'></a>

A small number of records have null values in `end_lat` or `end_lng`.  
These trips cannot be mapped or spatially analysed — they are removed from the dataset.

In [8]:
mask_end_na = df['end_lat'].isnull() | df['end_lng'].isnull()
df_end_na   = df[mask_end_na]

print(f"Rows with missing end coordinates : {len(df_end_na):,}")
print(f"  - end_lat missing               : {df_end_na['end_lat'].isna().sum():,}")
print(f"  - end_lng missing               : {df_end_na['end_lng'].isna().sum():,}")
print(f"  - also missing end_station_name : {df_end_na['end_station_name'].isna().sum():,}")

Rows with missing end coordinates : 5,896
  - end_lat missing               : 5,896
  - end_lng missing               : 5,896
  - also missing end_station_name : 5,896


In [9]:
df_cl        = df.dropna(subset=['end_lat', 'end_lng']).copy()
n_after_gps  = len(df_cl)

print(f"Removed  : {n_raw - n_after_gps:,} rows  ({(n_raw - n_after_gps)/n_raw*100:.3f}%)")
print(f"Remaining: {n_after_gps:,} rows")

Removed  : 5,896 rows  (0.101%)
Remaining: 5,842,807 rows


## 3.2 Electric Bikes and Missing Station Names <a id='ebike'></a>

Divvy's system rules differ by bike type:
- **Classic bikes** must be returned to a docking station to end the trip; missing station names indicate a potential system error.
- **Electric bikes** support *flexible locking* — they can be locked to any public rack without docking. Missing station names for e-bikes are **expected and valid** and must **not** be removed.

The cell below confirms that all records with null station names belong to electric bikes, validating this rule.

In [10]:
missing_start = df_cl['start_station_name'].isna()
missing_end   = df_cl['end_station_name'].isna()

print("Bike types with MISSING START station:")
print(df_cl.loc[missing_start, 'rideable_type'].value_counts().to_string())

print("\nBike types with MISSING END station:")
print(df_cl.loc[missing_end, 'rideable_type'].value_counts().to_string())

total_ebike  = (df_cl['rideable_type'] == 'electric_bike').sum()
ebike_no_start = (missing_start & (df_cl['rideable_type'] == 'electric_bike')).sum()
ebike_no_end   = (missing_end   & (df_cl['rideable_type'] == 'electric_bike')).sum()

print(f"\nE-bikes without start station : {ebike_no_start:,} / {total_ebike:,} ({ebike_no_start/total_ebike*100:.1f}%)")
print(f"E-bikes without end station   : {ebike_no_end:,} / {total_ebike:,} ({ebike_no_end/total_ebike*100:.1f}%)")

total_classic    = (df_cl['rideable_type'] == 'classic_bike').sum()
classic_no_start = (missing_start & (df_cl['rideable_type'] == 'classic_bike')).sum()
classic_no_end   = (missing_end   & (df_cl['rideable_type'] == 'classic_bike')).sum()

print(f"\nClassic bikes without start station : {classic_no_start:,} / {total_classic:,} ({classic_no_start/total_classic*100:.2f}%)")
print(f"Classic bikes without end station   : {classic_no_end:,} / {total_classic:,} ({classic_no_end/total_classic*100:.2f}%)")

Bike types with MISSING START station:


rideable_type
electric_bike    1249667

Bike types with MISSING END station:
rideable_type
electric_bike    1308364
classic_bike          74



E-bikes without start station : 1,249,667 / 3,902,512 (32.0%)
E-bikes without end station   : 1,308,364 / 3,902,512 (33.5%)



Classic bikes without start station : 0 / 1,940,295 (0.00%)
Classic bikes without end station   : 74 / 1,940,295 (0.00%)


## 3.3 Standardizing Categorical Values <a id='standardize'></a>

The `member_casual` column is normalized to lowercase and stripped of leading/trailing whitespace  
to prevent grouping errors caused by inconsistent casing (e.g. `'Member'` vs `'member'`).

In [11]:
df_cl['member_casual'] = df_cl['member_casual'].str.lower().str.strip()
print("Unique values in member_casual after standardization:", df_cl['member_casual'].unique())

Unique values in member_casual after standardization: ['member' 'casual']


## 3.4 Deduplication Check <a id='dedup'></a>

Each trip should have a unique `ride_id`. Duplicates could skew aggregated metrics  
(e.g. total usage counts, distance calculations). Any duplicate found is reported here.

In [12]:
ride_id_counts = df_cl['ride_id'].value_counts()
duplicates     = ride_id_counts[ride_id_counts > 1]

if duplicates.empty:
    print("No duplicate ride_id values found. ✓")
else:
    print(f"Duplicate ride IDs found: {len(duplicates):,}")
    print(duplicates.head())

Duplicate ride IDs found: 29
ride_id
C9050956F58561AD    2
5C56F4FB55D5B5F5    2
4B93DFFDA6B08DE5    2
D45C58C1A07DDA84    2
D28B306F00821227    2
Name: count, dtype: int64


## 3.5 Timestamp Consistency <a id='timestamps'></a>

Both timestamp columns are converted to `datetime64` and then filtered to remove any records  
where `ended_at ≤ started_at`. These represent clock errors, test entries, or data artifacts  
and would produce negative or zero trip durations.

In [13]:
# Convert to datetime (idempotent if already converted)
df_cl['started_at'] = pd.to_datetime(df_cl['started_at'])
df_cl['ended_at']   = pd.to_datetime(df_cl['ended_at'])

# Remove rows where trip ends before or at the same time as it starts
df_clean = df_cl[df_cl['started_at'] < df_cl['ended_at']].copy()
n_final  = len(df_clean)

print(f"Removed  : {n_after_gps - n_final:,} rows with ended_at ≤ started_at")
print(f"Remaining: {n_final:,} rows")

Removed  : 29 rows with ended_at ≤ started_at
Remaining: 5,842,778 rows


## 3.6 Cross-Month Rides <a id='xmonth'></a>

Rides whose `started_at` and `ended_at` fall in **different months** are removed.
These are typically very long rides (e.g. a bike left out overnight across a month boundary)
that would otherwise inject a spurious extra month into downstream visualizations.

In [14]:
n_before_xmonth = len(df_clean)

same_month_mask = (
    df_clean['started_at'].dt.year.eq(df_clean['ended_at'].dt.year) &
    df_clean['started_at'].dt.month.eq(df_clean['ended_at'].dt.month)
)
df_clean = df_clean.loc[same_month_mask].copy()
n_final  = len(df_clean)

removed_xmonth = n_before_xmonth - n_final
print(f"Rides before filter : {n_before_xmonth:,}")
print(f"Rides removed (cross-month): {removed_xmonth:,} "
      f"({removed_xmonth / n_before_xmonth * 100:.2f}%)")
print(f"Rides after filter  : {n_final:,}")

Rides before filter : 5,842,778
Rides removed (cross-month): 853 (0.01%)
Rides after filter  : 5,841,925


---
# 4. Cleaning Summary <a id='summary'></a>
[↑ back to top](#toc)

Overview of all cleaning steps, the number of rows removed at each stage, and the final dataset size.

In [15]:
steps = [
    ("Initial load — 12 monthly CSV files",                       None,                          n_raw),
    ("Remove rows with missing end GPS (end_lat / end_lng)",       n_raw - n_after_gps,           n_after_gps),
    ("Remove timestamp inconsistencies (ended_at ≤ started_at)",   n_after_gps - n_before_xmonth, n_before_xmonth),
    ("Remove cross-month rides (start/end in different months)",    removed_xmonth,                n_final),
]

header = f"{'Step':<62} {'Removed':>9} {'Remaining':>11}"
print(header)
print('─' * len(header))
for step, removed, remaining in steps:
    r = '—' if removed is None else f'{removed:,}'
    print(f"  {step:<60} {r:>9} {remaining:>11,}")

pct_lost = (n_raw - n_final) / n_raw * 100
print()
print(f"Total removed : {n_raw - n_final:,} rows ({pct_lost:.2f}% of raw data)")
print(f"Final dataset : {n_final:,} rows | {len(files)} monthly files | "
      f"{df_clean['member_casual'].nunique()} customer segments | "
      f"{df_clean['rideable_type'].nunique()} bike types")


Step                                                             Removed   Remaining
────────────────────────────────────────────────────────────────────────────────────
  Initial load — 12 monthly CSV files                                  —   5,848,703
  Remove rows with missing end GPS (end_lat / end_lng)             5,896   5,842,807
  Remove timestamp inconsistencies (ended_at ≤ started_at)            29   5,842,778
  Remove cross-month rides (start/end in different months)           853   5,841,925

Total removed : 6,778 rows (0.12% of raw data)


Final dataset : 5,841,925 rows | 12 monthly files | 2 customer segments | 2 bike types


---
# 5. Conclusions <a id='conclusions'></a>
[↑ back to top](#toc)

### Cleaning decisions summary

| Step | Decision | Rationale |
|------|----------|-----------|
| Missing end GPS | **Removed** | Cannot be spatially analysed or mapped |
| Missing station names (e-bikes) | **Retained** | By design — flexible locking does not require docking |
| Missing station names (classic bikes) | **None found** | No classic-bike records had null station names |
| Duplicate ride IDs | **None found** | No action required |
| Timestamp inconsistencies | **Removed** | Negative or zero durations indicate clock errors |

### Data quality assessment

| Dimension | Assessment |
|-----------|------------|
| **Completeness** | High — only GPS-null rows removed (< 0.2% of raw data) |
| **Consistency** | Good — categorical values standardized, timestamps validated |
| **Validity** | Good — coordinates within Chicago bounds, ride IDs unique |
| **Coverage** | 12 months (Jun 2025 – May 2026), two user segments, three bike types |

### Next step
The clean dataset `df_clean.csv` is ready for feature engineering and analysis.  
Continue with **`02_zone_data_processing.ipynb`** (zoning enrichment) and **`03_analysis_and_visualization.ipynb`** (EDA and insights).

---
# 6. Saving Outputs <a id='saving'></a>
[↑ back to top](#toc)

Two outputs are produced:
1. **`df_clean.csv`** — the clean, analysis-ready dataset saved to `../data/processed/`.
2. **`cyclistic_cleaning.html`** — an HTML export of this notebook (markdown + outputs only, no input code) saved to `../reports/`.

In [16]:
directory = os.path.join('../data/processed', 'df_clean.csv')
df_clean.to_csv(directory, index=False)

if os.path.exists(directory):
    size_mb = os.path.getsize(directory) / 1e6
    print(f"Saved: {os.path.abspath(directory)}  ({size_mb:.0f} MB)")

Saved: /Users/lorenzodigiacomo/Desktop/Ciclistic_analisi/data/processed/df_clean.csv  (1068 MB)


In [17]:
import nbformat
from nbconvert import HTMLExporter
from nbconvert.preprocessors import Preprocessor
import copy

notebook_path = '01_data_cleaning.ipynb'
output_html = '../reports/cyclistic_cleaning.html'

with open(notebook_path, 'r', encoding='utf-8') as f:
    notebook = nbformat.read(f, as_version=4)

notebook_filtered = copy.deepcopy(notebook)

filtered_cells = []
for cell in notebook_filtered.cells:
    if cell.cell_type == 'markdown':
        filtered_cells.append(cell)

    elif cell.cell_type == 'raw':
        # Convert raw → code cell so nbconvert renders it as a styled box
        # preserving indentation and spacing
        code_cell = nbformat.v4.new_code_cell(source=cell.source)
        code_cell.execution_count = None
        filtered_cells.append(code_cell)

    elif cell.cell_type == 'code' and cell.outputs:
        cell.source = ''
        cell.execution_count = None
        for output in cell.outputs:
            if hasattr(output, 'execution_count'):
                output.execution_count = None
        filtered_cells.append(cell)

notebook_filtered.cells = filtered_cells

html_exporter = HTMLExporter()
html_exporter.exclude_input_prompt = True
html_exporter.exclude_output_prompt = True 

(body, resources) = html_exporter.from_notebook_node(notebook_filtered)

# Inject CSS to style the raw-converted cells distinctively (optional)
custom_css = """
<style>
  /* Raw cells (converted to code cells) — neutral background to distinguish
     them from actual code outputs */
  .cell:has(.input:not(:empty)) .input_area {
      background-color: #f5f5f5;
      border-left: 3px solid #aaaaaa;
  }
</style>
"""
body = body.replace('</head>', custom_css + '</head>')

with open(output_html, 'w', encoding='utf-8') as f:
    f.write(body)

print(f"Successfully created {output_html} from {notebook_path}")
print(f"Original notebook {notebook_path} remains unchanged")
print(f"Cells exported: {len(notebook_filtered.cells)} "
      f"(markdown + raw as styled box + code outputs only)")

Successfully created ../reports/cyclistic_cleaning.html from 01_data_cleaning.ipynb
Original notebook 01_data_cleaning.ipynb remains unchanged
Cells exported: 31 (markdown + raw as styled box + code outputs only)
